## Modèles récurrents : apprendre l'allocation de Markowitz

1. **la cible** : ce que le modèle doit reproduire ;
2. **les trois architectures** V0, V1 et V2 ;
3. **les fonctions de perte** : entropie croisée et Sharpe ;
4. **le chargement des données** et la construction des séquences ;
5. **la normalisation** ;
6. **des repères chiffrés** : les scores que les modèles devront battre.

## 1. La cible : l'oracle de Markowitz

**Pour chaque date $t$, l'oracle est le portefeuille long-only qui a maximisé le Sharpe sur les 63 jours suivants (de $t+1$ à $t+63$).** Il vaut **100 % CASH** dans les rares cas (2,4 % des dates) où aucun portefeuille d'actions n'a battu le taux sans risque.

Le modèle doit reproduire cette allocation en n'utilisant que l'information disponible à $t$.

## 2. Les trois architectures

Tous les modèles prennent une décision à la date $t$ et sortent **16 poids** (15 actions + CASH) via un **softmax** : les poids sont positifs et somment à 1, exactement comme $Y_t$.

|  | **V0** | **V1** | **V2** |
|---|---|---|---|
| Ce que lit le LSTM | 1 séquence de 60 jours × 15 actions | 15 séquences de 52 semaines × 2 (action, CAC) | idem V1 |
| Tenseur d'entrée du LSTM | (B, 60, 15) | (B·15, 52, 2) | idem V1 |
| Poids du LSTM | un seul LSTM pour tout | **un seul LSTM partagé** par les 15 actions | idem V1 |
| Sortie du LSTM | 1 état pour l'ensemble du marché | 15 états, un par action | idem V1 |
| Ajouté avant la tête de sortie | rien | rien | 10 features de l'action + 10 features globales |
| Score du CASH | depuis l'état unique | depuis la moyenne des 15 états | depuis les features globales |
| Sortie | 16 poids (softmax) | 16 poids (softmax) | 16 poids (softmax) |

B = taille du batch (nombre de dates traitées ensemble).

### Points communs aux trois versions

- **Fenêtre fixe** : on donne toujours la même longueur (60 jours ou 52 semaines), à l'entraînement comme en inférence. Pour une autre longueur, on réentraîne et on compare sur la validation.
- **Plusieurs entrées → une sortie** : le LSTM lit toute la fenêtre (état initial à 0), et **seul l'état final** produit l'allocation. Il n'y a ni prédiction intermédiaire ni autorégression : le LSTM sert d'**encodeur** qui résume le passé.
- **Taille de l'état caché** $h$ (8, 16 ou 32) : un hyperparamètre, choisi sur la validation. À ne pas confondre avec la longueur de la séquence.

## 3. Les fonctions de perte

Notations pour un échantillon (une date $t$) :
- $\pi \in \mathbb{R}^{16}$ : poids prédits (sortie du softmax) ;
- $Y \in \mathbb{R}^{16}$ : poids de l'oracle ;
- $R \in \mathbb{R}^{16}$ : rendements réalisés de chaque actif entre $t+1$ et $t+63$ (`R_fwd`), et $R_{\text{cash}}$ celui du CASH.

### 3.1 Entropie croisée

$$
\mathcal{L}_{CE} = -\sum_{i=1}^{16} Y_i \log \pi_i
$$

### 3.2 Sharpe : optimiser directement la performance

Pour un batch de $B$ dates, le rendement en excès du portefeuille prédit est $r_b = \pi_b^\top R_b - R_{\text{cash},b}$, et :

$$
\mathcal{L}_{Sharpe} = -\,\frac{\operatorname{moyenne}_b(r_b)}{\operatorname{écart\text{-}type}_b(r_b)}
$$

- Cette perte récompense directement les allocations qui ont bien performé.
- Le Sharpe est estimé sur le batch : il faut des **batches d'au moins 128 dates**, sinon l'estimation est trop bruitée.
- Elle permet d'exploiter la partie **prévisible** du problème : le risque (la volatilité passée prédit bien la volatilité future), alors que les rendements futurs sont quasiment imprévisibles.

### 3.3 Hybride

$$
\mathcal{L} = \mathcal{L}_{CE} + \alpha \, \mathcal{L}_{Sharpe}, \qquad \alpha \in \{0 ;\, 0{,}1 ;\, 0{,}5\}
$$

$\alpha = 0$ : imitation pure. $\alpha > 0$ : imiter l'oracle tout en étant récompensé par la performance réalisée.

### 3.4 Critères de réussite

- Entropie croisée de validation inférieure à celle de la stratégie « toujours prédire la moyenne des cibles du train ». Sinon, le modèle n'a rien appris au-delà de la répartition moyenne.
- **Sharpe au moins comparable à celui du portefeuille équipondéré**, idéalement avec moins de turnover.

### Chargement des données

Fichiers produits par `data_generation/build_dataset.py`. Le découpage est déjà fait dans `meta.csv` : train 2006–2016, val 2017–2019, test 2020–2025 ; les dates `none` sont purgées (leur cible chevauche le bloc suivant).

In [23]:
from pathlib import Path
import numpy as np
import pandas as pd

DATA, N = Path("data/processed/N63_wmax1"), 63
read = lambda f: pd.read_csv(f, index_col=0, parse_dates=True)
X, Y, R_fwd, meta, rets = (read(DATA / f"{f}.csv") for f in ["X", "Y", "R_fwd", "meta", "returns"])
TICKERS = list(Y.columns[:-1])

idx = {s: np.flatnonzero(meta["split"].to_numpy() == s) for s in ["train", "val", "test"]}
for s, i in idx.items():
    print(f"{s:5s} : {len(i):4d} dates ({X.index[i[0]].date()} → {X.index[i[-1]].date()}), ~{len(i) // N} trimestres indépendants")

train : 2749 dates (2006-01-02 → 2016-10-03), ~43 trimestres indépendants
val   :  703 dates (2017-01-02 → 2019-10-02), ~11 trimestres indépendants
test  : 1474 dates (2020-01-02 → 2025-09-30), ~23 trimestres indépendants


Construction des entrées (rendements en log) :
- **V0** : les 60 derniers rendements journaliers des 15 actions → (dates, 60, 15) ;
- **V1/V2** : pour chaque action, 52 rendements hebdomadaires, avec ceux du CAC en 2ᵉ canal → (dates, 15, 52, 2). Le CAC est recalculé depuis `data/raw/prices.csv` ;
- **V2** : features par action (dates, 15, 10) et globales (dates, 10).

In [24]:
logr = np.log1p(rets.to_numpy())
pos = rets.index.get_indexer(X.index)                  # position de chaque date t dans le calendrier

# V0
seq_v0 = np.stack([logr[i - 59:i + 1] for i in pos])

# V1/V2 : CAC avec le même nettoyage que build_dataset.py
px = read("data/raw/prices.csv").dropna(how="all").ffill(limit=2).dropna(how="any")
cac = px["^FCHI"].pct_change().iloc[1:]
assert cac.index.equals(rets.index)
log_cac = np.log1p(cac.to_numpy())

def weekly(lr, n_weeks=52, step=5):
    # rendements log hebdomadaires, la dernière semaine finissant à t (semaines manquantes = 0)
    C = np.concatenate([np.zeros((1,) + lr.shape[1:]), lr.cumsum(axis=0)])
    out = np.zeros((len(pos), n_weeks) + lr.shape[1:])
    for k, i in enumerate(pos):
        ends = i + 1 - step * np.arange(n_weeks)[::-1]
        ok = ends >= step
        out[k, ok] = C[ends[ok]] - C[ends[ok] - step]
    return out

seq_v1 = np.stack([weekly(logr).transpose(0, 2, 1),
                   np.repeat(weekly(log_cac)[:, None, :], 15, axis=1)], axis=-1)

# V2 : features rangées par action, et globales
ASSET_FEATS = ["mom_5", "mom_21", "mom_63", "mom_126", "mom_252", "vol_21", "vol_63", "dvol_63", "dd_252", "corrmkt_63"]
feat_asset = np.stack([X[[f"{f}__{tk}" for f in ASSET_FEATS]].to_numpy() for tk in TICKERS], axis=1)
feat_glob = X[[c for c in X.columns if c.endswith("__global") or c.startswith(("market__", "macro__"))]].to_numpy()

print("seq_v0", seq_v0.shape, "| seq_v1", seq_v1.shape, "| feat_asset", feat_asset.shape, "| feat_glob", feat_glob.shape)

seq_v0 (5082, 60, 15) | seq_v1 (5082, 15, 52, 2) | feat_asset (5082, 15, 10) | feat_glob (5082, 10)


### Normalisation
Statistiques calculées **sur le train uniquement**. Les rendements des actions partagent **une seule échelle** (normaliser action par action effacerait leurs différences de volatilité, qui sont informatives). Les features par action sont normalisées par type de feature, les globales par colonne. $Y$ et `R_fwd` ne sont pas normalisés.

In [25]:
tr = idx["train"]
seq_v0_n = seq_v0 / seq_v0[tr].std()
seq_v1_n = seq_v1 / np.array([seq_v1[tr][..., 0].std(), seq_v1[tr][..., 1].std()])
feat_asset_n = (feat_asset - feat_asset[tr].mean(axis=(0, 1))) / feat_asset[tr].std(axis=(0, 1))
feat_glob_n = (feat_glob - feat_glob[tr].mean(axis=0)) / feat_glob[tr].std(axis=0)

for name, a in [("seq_v0", seq_v0_n), ("seq_v1", seq_v1_n), ("feat_asset", feat_asset_n), ("feat_glob", feat_glob_n)]:
    print(f"{name:10s} écart-type  train {a[tr].std():.2f} | val {a[idx['val']].std():.2f} | test {a[idx['test']].std():.2f}")

seq_v0     écart-type  train 1.00 | val 0.59 | test 0.92
seq_v1     écart-type  train 1.00 | val 0.64 | test 0.96
feat_asset écart-type  train 1.00 | val 0.77 | test 1.07
feat_glob  écart-type  train 1.00 | val 1.20 | test 1.52


Sharpe en excès du CASH, rééquilibrage tous les 63 jours, moyenne (± écart-type) sur les 63 dates de départ possibles.

In [26]:
def sharpe(W, block):
    i = idx[block]; W, R = W[i], R_fwd.to_numpy()[i]
    v = [((W[o::N] * R[o::N]).sum(1) - R[o::N, -1]) for o in range(N)]
    v = [x.mean() / x.std() * np.sqrt(252 / N) for x in v]
    return f"{np.mean(v):+.2f} ± {np.std(v):.2f}"

def ce(W, block):
    i = idx[block]
    return round(float(-(Y.to_numpy()[i] * np.log(W[i] + 1e-6)).sum(1).mean()), 3)

T = len(Y)
strats = {"Oracle": Y.to_numpy(),
          "Équipondéré": np.c_[np.full((T, 15), 1 / 15), np.zeros(T)],
          "Moyenne des cibles du train": np.tile(Y.to_numpy()[tr].mean(0), (T, 1))}
pd.DataFrame({k: {"Sharpe val": sharpe(W, "val"), "Sharpe test": sharpe(W, "test"),
                  "CE val": None if k == "Oracle" else ce(W, "val")} for k, W in strats.items()}).T

,Sharpe val,Sharpe test,CE val
Oracle,+4.47 ± 0.86,+4.93 ± 0.84,None
Équipondéré,+1.04 ± 0.15,+0.84 ± 0.28,2.724
Moyenne des cibles du train,+1.20 ± 0.18,+0.78 ± 0.29,2.642


### Grid search de V0 (LSTM et GRU)

On utilise le découpage déjà fait : **entraînement sur le train (2006–2016), choix sur la validation (2017–2019)**, une seule graine pour l'instant. Pour chaque combinaison : séquences de profondeur $L$ normalisées avec l'écart-type du train, entraînement avec l'entropie croisée, arrêt anticipé sur l'entropie croisée de validation, puis calcul du Sharpe de validation des poids prédits.

Repères (section précédente) : entropie croisée de validation de la « moyenne des cibles du train » = **2,642** ; Sharpe de validation de l'équipondéré = **1,04**.

In [27]:
import itertools, time
import torch
import torch.nn as nn

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device :", DEVICE)

device : cpu


In [28]:
# Architecture V0
class V0(nn.Module):
    def __init__(self, model="lstm", hidden=16, layers=1, dropout=0.0, n_assets=15):
        super().__init__()
        rnn = nn.LSTM if model == "lstm" else nn.GRU
        self.rec = rnn(
            input_size = n_assets,
            hidden_size=hidden,
            num_layers=layers,
            batch_first=True,
            dropout=dropout if layers > 1 else 0.0
        )
        self.drop = nn.Dropout(dropout)
        self.head = nn.Linear(hidden, n_assets +1)

    def forward(self,x):
        out, _ =self.rec(x)
        last = out[:,-1]
        return torch.softmax(self.head(self.drop(last)), dim=-1)   # (B, 16), log des poids

In [ ]:
def grid_search(grid, tr_i=None, va_i=None, seed=0, lr=1e-3, weight_decay=1e-4, batch=128, max_epochs=100, patience=5, cross_entropy=True):
    # Teste toutes les combinaisons de `grid` : entraînement sur tr_i, arrêt anticipé et score sur va_i
    # (par défaut : le train et la validation du découpage de base).
    tr_i = idx["train"] if tr_i is None else tr_i
    va_i = idx["val"] if va_i is None else va_i
    Yt = torch.tensor(Y.to_numpy(), dtype=torch.float32)
    R_val = R_fwd.to_numpy()[va_i]
    if cross_entropy:
        criterion = lambda p, y: -(y * torch.log(p + 1e-8)).sum(-1).mean()

    combos = list(itertools.product(*grid.values()))
    results = []
    for n, values in enumerate(combos, 1):
        cfg = dict(zip(grid.keys(), values))
        t0 = time.time()

        # séquences : les L derniers rendements log journaliers, normalisés avec l'écart-type du train
        L = cfg.get("L", 60)                                          # profondeur : 60 jours si absente de la grille
        S = np.stack([logr[i - L + 1:i + 1] for i in pos]).astype(np.float32)
        S /= S[tr_i].std()
        x_tr, x_va = torch.tensor(S[tr_i]), torch.tensor(S[va_i]).to(DEVICE)
        y_tr, y_va = Yt[tr_i], Yt[va_i].to(DEVICE)

        torch.manual_seed(seed)
        model = V0(cfg["model"], cfg["hidden"], cfg["layers"], cfg["dropout"]).to(DEVICE)
        opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

        best, best_ep, best_state, wait = float("inf"), 0, None, 0
        for ep in range(max_epochs):
            model.train()
            perm = torch.randperm(len(tr_i))                      # mélange à l'intérieur du train
            for b in range(0, len(perm), batch):
                j = perm[b:b + batch]
                loss = criterion(model(x_tr[j].to(DEVICE)), y_tr[j].to(DEVICE))
                opt.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step()

            model.eval()
            with torch.no_grad():
                val_loss = criterion(model(x_va), y_va).item()
            if val_loss < best - 1e-4:                            # arrêt anticipé
                best, best_ep, wait = val_loss, ep + 1, 0
                best_state = {k: t.clone() for k, t in model.state_dict().items()}
            else:
                wait += 1
                if wait >= patience:
                    break
        model.load_state_dict(best_state)

        # Sharpe de validation : rééquilibrage tous les N jours, moyenne sur les N dates de départ
        with torch.no_grad():
            W = model(x_va).cpu().numpy()
        sharpes = []
        for o in range(N):
            ex = (W[o::N] * R_val[o::N]).sum(1) - R_val[o::N, -1]
            sharpes.append(ex.mean() / ex.std() * np.sqrt(252 / N))

        results.append({**cfg, "val_loss": best, "val_sharpe": np.mean(sharpes), "best_epoch": best_ep,
                        "n_params": sum(p.numel() for p in model.parameters()), "sec": round(time.time() - t0, 1)})
        print(f"[{n}/{len(combos)}] {cfg} → loss {best:.3f} | Sharpe {np.mean(sharpes):+.2f} "
              f"| {best_ep} époques | {time.time() - t0:.0f} s")

    return pd.DataFrame(results).sort_values("val_ce").reset_index(drop=True)

In [ ]:
GRID = {
    "model":   ["lstm", "gru"],
    "hidden":  [8, 16, 32],     # taille de l'état caché
    "layers":  [1, 2],          # nombre de couches
    "dropout": [0.0, 0.2],
    "L":       [20, 60, 120],   # profondeur historique (jours)
}
res_v0 = grid_search(GRID, seed=0)
res_v0.to_csv("models/lstm/results/grid_v0_seed0.csv", index=False)
res_v0.round(3).head(15)

[1/72] {'model': 'lstm', 'hidden': 8, 'layers': 1, 'dropout': 0.0, 'L': 20} → CE 2.653 | Sharpe +1.20 | 7 époques | 2 s
[2/72] {'model': 'lstm', 'hidden': 8, 'layers': 1, 'dropout': 0.0, 'L': 60} → CE 2.653 | Sharpe +1.20 | 7 époques | 3 s
[3/72] {'model': 'lstm', 'hidden': 8, 'layers': 1, 'dropout': 0.0, 'L': 120} → CE 2.653 | Sharpe +1.20 | 7 époques | 4 s
[4/72] {'model': 'lstm', 'hidden': 8, 'layers': 1, 'dropout': 0.2, 'L': 20} → CE 2.653 | Sharpe +1.19 | 7 époques | 2 s
[5/72] {'model': 'lstm', 'hidden': 8, 'layers': 1, 'dropout': 0.2, 'L': 60} → CE 2.653 | Sharpe +1.19 | 7 époques | 3 s
[6/72] {'model': 'lstm', 'hidden': 8, 'layers': 1, 'dropout': 0.2, 'L': 120} → CE 2.653 | Sharpe +1.19 | 7 époques | 4 s
[7/72] {'model': 'lstm', 'hidden': 8, 'layers': 2, 'dropout': 0.0, 'L': 20} → CE 2.654 | Sharpe +1.19 | 5 époques | 2 s
[8/72] {'model': 'lstm', 'hidden': 8, 'layers': 2, 'dropout': 0.0, 'L': 60} → CE 2.654 | Sharpe +1.19 | 5 époques | 3 s
[9/72] {'model': 'lstm', 'hidden': 8, 

### Validation croisée walk-forward

Le grid search ci-dessus choisit sur une seule période (2017–2019). Ici, on répète la sélection sur **4 plis** : à chaque pli, on s'entraîne sur tout le passé et on valide sur les 2 années suivantes. La purge est la même que dans le découpage de base : un échantillon d'entraînement n'est gardé que si sa cible finit avant le début de la validation. Le test (2020–2025) n'est pas utilisé.

| Pli | Train (dates t) | Validation |
|---|---|---|
| 1 | 2006 → 2011 | 2012–2013 |
| 2 | 2006 → 2013 | 2014–2015 |
| 3 | 2006 → 2015 | 2016–2017 |
| 4 | 2006 → 2017 | 2018–2019 |

La profondeur historique est fixée à 60 jours. Le score d'une configuration est la moyenne sur les 4 plis ; on la compare à la référence « moyenne des cibles du train » de chaque pli.

In [21]:
label_end = pd.to_datetime(meta["label_end"])
FOLDS = {}
for y0 in [2012, 2014, 2016, 2018]:
    v_start, v_end = pd.Timestamp(f"{y0}-01-01"), pd.Timestamp(f"{y0 + 2}-01-01")
    tr_i = np.flatnonzero((X.index >= "2006-01-01") & (label_end < v_start).to_numpy())   # purge
    va_i = np.flatnonzero((X.index >= v_start) & (label_end < v_end).to_numpy())
    FOLDS[f"{y0}-{y0 + 1}"] = (tr_i, va_i)

# référence de chaque pli : prédire toujours la moyenne des cibles du train du pli
ref_ce = {name: -(Y.to_numpy()[va] * np.log(Y.to_numpy()[tr].mean(0) + 1e-8)).sum(1).mean() for name, (tr, va) in FOLDS.items()}
for name, (tr, va) in FOLDS.items():
    print(f"pli {name} : train {len(tr)} dates, validation {len(va)} dates, CE de référence {ref_ce[name]:.3f}")

pli 2012-2013 : train 1472 dates, validation 446 dates, CE de référence 2.801
pli 2014-2015 : train 1981 dates, validation 448 dates, CE de référence 2.644
pli 2016-2017 : train 2492 dates, validation 449 dates, CE de référence 2.811
pli 2018-2019 : train 3004 dates, validation 448 dates, CE de référence 2.629


In [30]:
GRID_WF = {"model": ["lstm", "gru"], "hidden": [8, 16, 32], "layers": [1, 2], "dropout": [0.0, 0.2]}

res_wf = pd.concat([grid_search(GRID_WF, tr_i, va_i).assign(fold=name) for name, (tr_i, va_i) in FOLDS.items()])
res_wf["gain_vs_ref"] = res_wf["fold"].map(ref_ce) - res_wf["val_ce"]          # > 0 : mieux que la référence
res_wf.to_csv("models/lstm/results/walk_forward_v0_seed0.csv", index=False)

summary_wf = (res_wf.groupby(list(GRID_WF))
              .agg(val_ce=("val_ce", "mean"), ce_std=("val_ce", "std"), gain_vs_ref=("gain_vs_ref", "mean"),
                   val_sharpe=("val_sharpe", "mean"), epochs=("best_epoch", "median"))
              .sort_values("val_ce"))
summary_wf.round(3)

[1/24] {'model': 'lstm', 'hidden': 8, 'layers': 1, 'dropout': 0.0} → CE 2.697 | Sharpe +1.84 | 8 époques | 2 s
[2/24] {'model': 'lstm', 'hidden': 8, 'layers': 1, 'dropout': 0.2} → CE 2.699 | Sharpe +1.84 | 8 époques | 1 s
[3/24] {'model': 'lstm', 'hidden': 8, 'layers': 2, 'dropout': 0.0} → CE 2.755 | Sharpe +1.73 | 1 époques | 1 s
[4/24] {'model': 'lstm', 'hidden': 8, 'layers': 2, 'dropout': 0.2} → CE 2.755 | Sharpe +1.73 | 1 époques | 2 s
[5/24] {'model': 'lstm', 'hidden': 16, 'layers': 1, 'dropout': 0.0} → CE 2.780 | Sharpe +1.70 | 6 époques | 2 s
[6/24] {'model': 'lstm', 'hidden': 16, 'layers': 1, 'dropout': 0.2} → CE 2.780 | Sharpe +1.70 | 6 époques | 2 s
[7/24] {'model': 'lstm', 'hidden': 16, 'layers': 2, 'dropout': 0.0} → CE 2.730 | Sharpe +1.79 | 1 époques | 2 s
[8/24] {'model': 'lstm', 'hidden': 16, 'layers': 2, 'dropout': 0.2} → CE 2.729 | Sharpe +1.79 | 1 époques | 2 s
[9/24] {'model': 'lstm', 'hidden': 32, 'layers': 1, 'dropout': 0.0} → CE 2.773 | Sharpe +1.76 | 1 époques | 

val_ce  ce_std  gain_vs_ref  val_sharpe  epochs
model hidden layers dropout                                                 
lstm  16     2      0.0       2.681   0.080        0.040       1.624     2.0
      8      1      0.2       2.682   0.068        0.039       1.618    12.5
      16     2      0.2       2.686   0.074        0.036       1.636     2.5
      8      1      0.0       2.687   0.061        0.034       1.593     7.5
      32     1      0.0       2.699   0.077        0.022       1.583     3.0
                    0.2       2.700   0.076        0.021       1.576     3.5
gru   8      1      0.0       2.700   0.067        0.021       1.551     6.0
                    0.2       2.702   0.065        0.019       1.547     7.0
      16     2      0.0       2.703   0.078        0.018       1.601     3.5
lstm  16     1      0.2       2.704   0.078        0.017       1.566     6.0
      32     2      0.0       2.704   0.080        0.017       1.575     2.0
gru   16     2      0.2       2.704   0.073        0.017       1.592     4.5
lstm  8      2      0.0       2.704   0.074        0.017       1.551     2.5
                    0.2       2.704   0.074        0.017       1.550     3.0
      16     1      0.0       2.706   0.075        0.015       1.553     5.5
gru   32     2      0.2       2.707   0.078        0.014       1.557     3.0
lstm  32     2      0.2       2.708   0.075        0.013       1.577     2.0
gru   32     2      0.0       2.709   0.075        0.012       1.565     2.5
             1      0.0       2.711   0.083        0.010       1.586     2.5
                    0.2       2.711   0.082        0.010       1.581     2.5
      16     1      0.0       2.712   0.077        0.010       1.555     5.0
      8      2      0.0       2.714   0.079        0.007       1.555     6.5
      16     1      0.2       2.714   0.074        0.007       1.555     5.5
      8      2      0.2       2.717   0.075        0.004       1.552     8.0

Les modeles apprennent a peine plus que plus que la répartition moyenne des cibles, mais très peu, et le choix des hyperparamètres ne compte presque pas.